In [ ]:
import sys, glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from gtsfm.common.depth_provider import DepthProvider

SEQ = 'office0'
DATA_ROOT = ''
DATA = f'{DATA_ROOT}/{SEQ}'
DEPTH_DIR = f'{DATA}/results'
DEPTH_SCALE = 6553.5
FRAME_IDX = 0

rgb_paths = sorted(glob.glob(f'{DATA}/results/frame*.jpg'))
depth_paths = sorted(glob.glob(f'{DEPTH_DIR}/depth*.png'))

rgb = cv2.cvtColor(cv2.imread(rgb_paths[FRAME_IDX]), cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
depth_m = cv2.imread(depth_paths[FRAME_IDX], cv2.IMREAD_UNCHANGED).astype(np.float64) / DEPTH_SCALE

kps = cv2.SIFT_create().detect(gray)
uvs = np.array([kp.pt for kp in kps])  # (N,2): col=u, row=v
print(f'SIFT keypoints: {len(uvs)}')

In [ ]:
PATCH_RADIUS = 5
GAP_THRESH = 0.15
AMBIGUITY_THRESH = 0.20

dp = DepthProvider(
    depth_map_dir=DEPTH_DIR,
    image_fnames={0: rgb_paths[FRAME_IDX]},
    depth_min=0.1, depth_max=10.0,
    depth_scale=DEPTH_SCALE,
    compute_hypotheses=True,
    patch_radius=PATCH_RADIUS,
    gap_thresh=GAP_THRESH,
    ambiguity_thresh=AMBIGUITY_THRESH,
)
samples = [dp.get_depth(0, u, v) for u, v in uvs]
scores = np.array([s.score if s else 0.0 for s in samples])
ambiguous = np.array([s.ambiguous if s else False for s in samples])
valid = np.array([s is not None for s in samples])
print(f'Valid: {valid.sum()} / {len(uvs)},  Ambiguous: {ambiguous.sum()} ({100*ambiguous.mean():.3f}%)')

In [ ]:
# Viz 1: score heatmap on depth map + flagged kps on RGB
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(depth_m, cmap='plasma', vmin=0, vmax=5)
sc = axes[0].scatter(uvs[valid,0], uvs[valid,1], c=scores[valid], cmap='hot', s=8, vmin=0, vmax=0.5)
plt.colorbar(sc, ax=axes[0], label='ambiguity score')
axes[0].set_title(f'Scores (R={PATCH_RADIUS}, gap={GAP_THRESH})')
axes[1].imshow(rgb)
axes[1].scatter(uvs[~ambiguous,0], uvs[~ambiguous,1], c='cyan', s=5, label='unimodal')
axes[1].scatter(uvs[ambiguous, 0], uvs[ambiguous, 1], c='red',  s=20, label='ambiguous')
axes[1].legend()
axes[1].set_title(f'Flagged: {ambiguous.sum()} / {len(uvs)}')
plt.tight_layout(); plt.show()

In [ ]:
# Viz 2: histogram of kp distance to nearest depth edge
depth_u8 = np.clip(depth_m / 5.0 * 255, 0, 255).astype(np.uint8)
edges    = cv2.Canny(depth_u8, 10, 30)
dist_map = cv2.distanceTransform((255 - edges), cv2.DIST_L2, 5)
kp_dists = np.array([
    dist_map[int(np.clip(round(v), 0, dist_map.shape[0]-1)),
             int(np.clip(round(u), 0, dist_map.shape[1]-1))]
    for u, v in uvs
])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(edges, cmap='gray')
axes[0].scatter(uvs[:,0], uvs[:,1], c='red', s=4, alpha=0.5)
axes[0].set_title('Depth edges + SIFT kps')
axes[1].hist(kp_dists, bins=50, color='steelblue', edgecolor='k')
axes[1].axvline(PATCH_RADIUS, color='red', linestyle='--', label=f'patch_radius={PATCH_RADIUS}')
axes[1].set_xlabel('Distance to nearest depth edge (px)')
axes[1].set_ylabel('# keypoints')
axes[1].set_title('How close do SIFT kps land to depth edges?')
axes[1].legend()
print(f'Kps within {PATCH_RADIUS}px of a depth edge: {100*(kp_dists <= PATCH_RADIUS).mean():.1f}%')
plt.tight_layout(); plt.show()

In [ ]:
# Viz 3: % flagged vs ambiguity_thresh
# Re-run with thresh=0 to collect raw scores, then threshold in numpy
dp_raw = DepthProvider(
    depth_map_dir=DEPTH_DIR,
    image_fnames={0: rgb_paths[FRAME_IDX]},
    depth_min=0.1, depth_max=10.0,
    depth_scale=DEPTH_SCALE,
    compute_hypotheses=True,
    patch_radius=PATCH_RADIUS,
    gap_thresh=0.0,
    ambiguity_thresh=0.0,
)
raw_scores  = np.array([s.score if s else 0.0 for s in [dp_raw.get_depth(0, u, v) for u, v in uvs]])
thresholds  = np.linspace(0.0, 0.5, 50)
pct_flagged = [100.0 * (raw_scores >= t).mean() for t in thresholds]
plt.figure(figsize=(7, 4))
plt.plot(thresholds, pct_flagged)
plt.axvline(AMBIGUITY_THRESH, color='red', linestyle='--', label=f'current thresh={AMBIGUITY_THRESH}')
plt.xlabel('ambiguity_thresh')
plt.ylabel('% keypoints flagged')
plt.title('Sensitivity of ambiguous fraction to threshold')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Funnel stage 2: matching attrition
# Load a second frame and run SIFT + TwoWayMatcher to see how many ambiguous kps survive matching.

FRAME_IDX2 = 1  # adjacent frame

rgb2  = cv2.cvtColor(cv2.imread(rgb_paths[FRAME_IDX2]), cv2.COLOR_BGR2RGB)
gray2 = cv2.cvtColor(rgb2, cv2.COLOR_RGB2GRAY)

sift_obj = cv2.SIFT_create(nfeatures=5000)
kps1, desc1 = sift_obj.detectAndCompute(gray,  None)
kps2, desc2 = sift_obj.detectAndCompute(gray2, None)
uvs1 = np.array([kp.pt for kp in kps1])

# Score ambiguity on frame 0 kps (same as before, reuse dp)
samples1   = [dp.get_depth(0, u, v) for u, v in uvs1]
ambig1     = np.array([s.ambiguous if s else False for s in samples1])
valid1     = np.array([s is not None for s in samples1])
print(f'Frame 0: {len(uvs1)} kps, {ambig1.sum()} ambiguous ({100*ambig1.mean():.2f}%)')

# Mutual NN + ratio test (mirrors TwoWayMatcher with ratio_test_threshold=0.8)
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

def ratio_match_oneway(d1, d2, thresh=0.8):
    matches = bf.knnMatch(d1, d2, k=2)
    return {m.queryIdx: m.trainIdx for m, n in matches if m.distance <= thresh * n.distance}

fwd = ratio_match_oneway(desc1, desc2)
bwd = ratio_match_oneway(desc2, desc1)
mutual = np.array([(i1, i2) for i1, i2 in fwd.items() if bwd.get(i2) == i1], dtype=np.int32)
print(f'Matches after mutual NN + ratio test: {len(mutual)}')

matched_i1 = set(mutual[:, 0]) if len(mutual) else set()
survived_matching = np.array([i in matched_i1 for i in range(len(uvs1))])

n_ambig_before = ambig1.sum()
n_ambig_after_match = (ambig1 & survived_matching).sum()
print(f'Ambiguous kps surviving matching: {n_ambig_after_match} / {n_ambig_before} '
      f'({100*n_ambig_after_match/max(n_ambig_before,1):.1f}%)')

In [ ]:
# Funnel stage 3: RANSAC attrition
# Run essential matrix RANSAC on the matched pairs and check ambiguous survival.
import json

# Load Replica intrinsics from cam_params.json
with open(f'{DATA}/cam_params.json') as f:
    cam = json.load(f)
fx, fy, cx, cy = cam['fx'], cam['fy'], cam['cx'], cam['cy']
K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)

pts1 = uvs1[mutual[:, 0]]
pts2 = np.array([kps2[i].pt for i in mutual[:, 1]])

# Normalize by K (matches what Ransac.estimate_E does)
pts1_norm = cv2.undistortPoints(pts1.reshape(-1,1,2), K, None).reshape(-1,2)
pts2_norm = cv2.undistortPoints(pts2.reshape(-1,1,2), K, None).reshape(-1,2)

E, inlier_mask = cv2.findEssentialMat(
    pts1_norm, pts2_norm,
    np.eye(3),
    method=cv2.USAC_ACCURATE,
    threshold=1.0 / fx,   # estimation_threshold_px=1.0 (default in unified.yaml)
    prob=0.999999,
)
inlier_mask = inlier_mask.ravel().astype(bool)
print(f'RANSAC inliers: {inlier_mask.sum()} / {len(mutual)} ({100*inlier_mask.mean():.1f}%)')

# Map inlier mask back to original kp indices in frame 0
inlier_i1 = set(mutual[inlier_mask, 0])
survived_ransac = np.array([i in inlier_i1 for i in range(len(uvs1))])

n_ambig_after_ransac = (ambig1 & survived_ransac).sum()
print(f'Ambiguous kps surviving RANSAC:   {n_ambig_after_ransac} / {n_ambig_before} '
      f'({100*n_ambig_after_ransac/max(n_ambig_before,1):.1f}%)')

# Funnel summary
print('\n--- Funnel summary (one pair) ---')
print(f'  Detected (ambiguous)  : {n_ambig_before} / {len(uvs1)}  ({100*ambig1.mean():.2f}%)')
print(f'  After matching        : {n_ambig_after_match} / {n_ambig_before}  ({100*n_ambig_after_match/max(n_ambig_before,1):.1f}% survival)')
print(f'  After RANSAC          : {n_ambig_after_ransac} / {n_ambig_after_match}  ({100*n_ambig_after_ransac/max(n_ambig_after_match,1):.1f}% survival)')

In [ ]:
# Multi-pair funnel: aggregate over many pairs sampled at STRIDE
# Mirrors what the actual pipeline sees (stride=10 means pairs of every 10th frame).
STRIDE    = 10
RATIO     = 0.8
EST_THRESH_PX = 1.0   # estimation_threshold_px from unified.yaml

frame_indices = list(range(0, len(rgb_paths), STRIDE))
pairs = [(frame_indices[i], frame_indices[i+1]) for i in range(len(frame_indices)-1)]

totals = dict(detected=0, ambig_detected=0, ambig_after_match=0, ambig_after_ransac=0,
              all_after_match=0, all_after_ransac=0)

sift_obj = cv2.SIFT_create(nfeatures=5000)

for idx_a, idx_b in pairs:
    imgA = cv2.cvtColor(cv2.imread(rgb_paths[idx_a]), cv2.COLOR_BGR2GRAY)
    imgB = cv2.cvtColor(cv2.imread(rgb_paths[idx_b]), cv2.COLOR_BGR2GRAY)
    kpA, dA = sift_obj.detectAndCompute(imgA, None)
    kpB, dB = sift_obj.detectAndCompute(imgB, None)
    if len(kpA) == 0 or len(kpB) == 0:
        continue
    uvsA = np.array([kp.pt for kp in kpA])

    # Ambiguity scoring on frame A
    dp_pair = DepthProvider(
        depth_map_dir=DEPTH_DIR,
        image_fnames={idx_a: rgb_paths[idx_a]},
        depth_min=0.1, depth_max=10.0,
        depth_scale=DEPTH_SCALE,
        compute_hypotheses=True,
        patch_radius=PATCH_RADIUS,
        gap_thresh=GAP_THRESH,
        ambiguity_thresh=AMBIGUITY_THRESH,
    )
    samps = [dp_pair.get_depth(idx_a, u, v) for u, v in uvsA]
    ambigA = np.array([s.ambiguous if s else False for s in samps])

    # Stage 2: matching
    fwd = ratio_match_oneway(dA, dB, thresh=RATIO)
    bwd = ratio_match_oneway(dB, dA, thresh=RATIO)
    mutual_pairs = np.array([(i1, i2) for i1, i2 in fwd.items() if bwd.get(i2) == i1], dtype=np.int32)
    if len(mutual_pairs) == 0:
        continue
    matched_set = set(mutual_pairs[:, 0])
    survived_match = np.array([i in matched_set for i in range(len(uvsA))])

    # Stage 3: RANSAC
    ptsA = uvsA[mutual_pairs[:, 0]]
    ptsB = np.array([kpB[i].pt for i in mutual_pairs[:, 1]])
    ptsA_n = cv2.undistortPoints(ptsA.reshape(-1,1,2), K, None).reshape(-1,2)
    ptsB_n = cv2.undistortPoints(ptsB.reshape(-1,1,2), K, None).reshape(-1,2)
    try:
        E, mask = cv2.findEssentialMat(ptsA_n, ptsB_n, np.eye(3),
                                        method=cv2.USAC_ACCURATE,
                                        threshold=EST_THRESH_PX/fx, prob=0.999999)
        if mask is None:
            continue
        mask = mask.ravel().astype(bool)
    except cv2.error:
        continue
    inlier_set = set(mutual_pairs[mask, 0])
    survived_ransac = np.array([i in inlier_set for i in range(len(uvsA))])

    totals['detected']          += len(uvsA)
    totals['ambig_detected']    += ambigA.sum()
    totals['ambig_after_match'] += (ambigA & survived_match).sum()
    totals['ambig_after_ransac']+= (ambigA & survived_ransac).sum()
    totals['all_after_match']   += survived_match.sum()
    totals['all_after_ransac']  += survived_ransac.sum()

print(f'Pairs processed: {len(pairs)}  (stride={STRIDE})')
print(f'\n--- Aggregate funnel ---')
n_d = totals['ambig_detected']
n_m = totals['ambig_after_match']
n_r = totals['ambig_after_ransac']
print(f'  Ambig detected     : {n_d} / {totals["detected"]}  ({100*n_d/max(totals["detected"],1):.2f}%)')
print(f'  Ambig after match  : {n_m} / {n_d}  ({100*n_m/max(n_d,1):.1f}% of ambig survive matching)')
print(f'  Ambig after RANSAC : {n_r} / {n_m}  ({100*n_r/max(n_m,1):.1f}% of matched-ambig survive RANSAC)')
print(f'\n  All kps after match  : {totals["all_after_match"]} / {totals["detected"]}  ({100*totals["all_after_match"]/max(totals["detected"],1):.1f}%)')
print(f'  All kps after RANSAC : {totals["all_after_ransac"]} / {totals["detected"]}  ({100*totals["all_after_ransac"]/max(totals["detected"],1):.1f}%)')